In [1]:
!pip install ultralytics
!pip install torchinfo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.1 MB/s eta 0:00:00


In [2]:
import cv2
import numpy as np
from ultralytics import YOLO



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# Load model (downloads weights on first run)
model = YOLO("yolov8n.pt")

In [ ]:
# Run torchinfo summary
pt_model = model.model

In [ ]:
#Load images from the MS COCO folder
# Raw predictions from YOLO model. Make sure the img_tensor is formatted accordingly
raw = model.model(img_tensor)

In [ ]:
if isinstance(raw, (list, tuple)):
    for i, t in enumerate(raw):
        print(i, type(t), getattr(t, "shape", None))
else:
    print(type(raw), raw.shape)

In [ ]:
# Run inference
results = model.predict(source=img, conf=0.25, iou=0.45, verbose=False)

In [ ]:
r = results[0]
names = model.names  # class id -> name
print(names)
print(r.boxes)

In [ ]:
# Extract detections
# r.boxes.xyxy: (N,4), r.boxes.conf: (N,), r.boxes.cls: (N,)
xyxy = r.boxes.xyxy.cpu().numpy() if r.boxes is not None else np.zeros((0, 4))
scores = r.boxes.conf.cpu().numpy() if r.boxes is not None else np.zeros((0,))
cls_ids = r.boxes.cls.cpu().numpy().astype(int) if r.boxes is not None else np.zeros((0,), dtype=int)
print(scores)
print(cls_ids)

In [ ]:
import os

image_parent_dir = 'coco_val_30_with_gt'
image_dir = os.path.join(image_parent_dir, 'images') # Point to the 'images' subdirectory


all_files = os.listdir(image_dir)

image_files = []
image_extensions = ('.jpg', '.jpeg', '.png')

for file_name in all_files:
    if file_name.lower().endswith(image_extensions):
        image_files.append(os.path.join(image_dir, file_name))

print(f"Found {len(image_files)} image files.")
print("30 image files:")
for i, img_file in enumerate(image_files[:30]):
    print(f"  {i+1}. {img_file}")


In [ ]:
import matplotlib.pyplot as plt

image_files_sel = image_files[2:4]
for img_path in image_files_sel:
    print(f"Processing image: {img_path}")


    # Load image (BGR)
    img = cv2.imread(img_path)
    if img is None:
        print(f"Warning: Could not read image {img_path}. Skipping.")
        continue

    # Run inference
    results = model.predict(source=img, conf=0.25, iou=0.45, verbose=False)

    r = results[0]

    # Extract detections
    xyxy = r.boxes.xyxy.cpu().numpy() if r.boxes is not None else np.zeros((0, 4))
    scores = r.boxes.conf.cpu().numpy() if r.boxes is not None else np.zeros((0,))
    cls_ids = r.boxes.cls.cpu().numpy().astype(int) if r.boxes is not None else np.zeros((0,), dtype=int)

    # Draw boxes
    vis = img.copy() # Make a copy to draw on, keep original img if needed
    for (x1, y1, x2, y2), s, c in zip(xyxy, scores, cls_ids):
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
        label = f"{names[c]} {s:.2f}"

        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(vis, label, (x1, max(0, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Display the image with detections
    plt.figure(figsize=(12, 9))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"Detections for {os.path.basename(img_path)}")
    plt.axis("off")
    plt.show()


In [ ]:

!pip -q install scikit-learn

import os
import numpy as np
import cv2
from tqdm import tqdm
from sklearn.metrics import average_precision_score

image_parent_dir = "/content/gdrive/MyDrive/coco_val_30_with_gt"
image_dir = os.path.join(image_parent_dir, "images")
gt_yolo_dir = os.path.join(image_parent_dir, "labels_yolo")

image_extensions = (".jpg", ".jpeg", ".png")
image_files = sorted([
    os.path.join(image_dir, f)
    for f in os.listdir(image_dir)
    if f.lower().endswith(image_extensions)
])

image_files_sel = image_files[:30]   # or any subset
CONF_THRES = 0.001
IOU_NMS = 0.1

C = len(names)
N = len(image_files_sel)

def read_yolo_classes(txt_path):
    if not os.path.isfile(txt_path):
        return set()
    s = set()
    with open(txt_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 1:
                s.add(int(float(parts[0])))
    return s

#Compute mAP using average_precision_score function
print(f"Multi-label mAP (over GT-present classes) = {mAP:.4f}")

